# 3D magnetics spatial fit — DIII-D 154551

A worked example of the **SLCONTOUR-style quasi-stationary spatial fit** (VISION.md §4.1),
run **locally** — off OMFIT and off MDSplus — using the ported OMFIT magnetics scripts in this
directory and the example netCDF data in `../data/154551/`.

**Pipeline:** `load → prep → fit → plot`, the local translation of the OMFIT
`fetch → prep → fit → plot` workflow:

| local module | OMFIT script |
|---|---|
| `io_data.load_shot` | `SCRIPTS/fetch_magnetics.py` (here just a loader) |
| `prep.prepare` | `SCRIPTS/prep_magnetics.py` |
| `fit.fit` | `SCRIPTS/fit_magnetics.py` |
| `run.run_steps` | `SCRIPTS/run_magnetics.py` |
| `plots.*` | `PLOTS/plot_magnetics_*.py` |

Shot **154551** is VISION's reference case: *"LFS Bp — a rotating n=1 that slows and locks; first
few singular values ≈ 98% energy."* We fit the LFS-midplane toroidal Bp array (`MPID66M*`, 10
sensors) for toroidal mode numbers n = 1, 2, 3.

In [ ]:
import sys
sys.path.insert(0, '.')  # make the local modules importable

import numpy as np
import matplotlib.pyplot as plt

from io_data import load_shot
from run import run_steps
import plots

%matplotlib inline

## 1. Load the shot (the "fetch" step)

The data was already fetched from MDSplus and saved to netCDF. `load_shot` reads the `RAW`,
`PLASMA_PARAMS` and `COUPLING` Datasets — the same structures `fetch_magnetics.py` builds. The
per-channel sensor geometry (`*_end1/2`) is already present, so `init_magnetics.py` is not needed.

In [ ]:
sd = load_shot(154551)
print(f'shot {sd.shot}  device {sd.device}')
print('RAW channels:', sd.raw.sizes['channel'], ' time samples:', sd.raw.sizes['time'])
print('helicity:', sd.plasma.attrs['helicity'])

# how many LFS-midplane Bp sensors carry good data?
from io_data import valid_channels
lfs = valid_channels(sd.raw, 'Bp_LFS_midplane', sd.device)
print(f'\nBp_LFS_midplane (MPID66M*): {len(lfs)} valid sensors  -> resolves |n| <= {(len(lfs)-1)//2}')
print(lfs)

### Named sensor / coil subsets

Any of these names can be passed as `channel_filter` (they live in the bundled
`DATA/DIII-D/` tables). The sensor and coil tables are merged.

In [ ]:
from io_data import available_subsets
for name, regex in available_subsets('DIII-D').items():
    print(f'{name:24s} {regex}')

## 2. Sensor geometry

The toroidal array sits at the LFS midplane (`theta ≈ 0`), spread around the torus in `phi`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
plots.plot_sensors(sd.raw, 'MPID66M.*', geometry='rz', ax=ax[0])
ax[0].set_title('Cross-section (R–z)')
plots.plot_sensors(sd.raw, 'MPID66M.*', geometry='cylindrical', ax=ax[1])
ax[1].set_title('Unrolled (phi–theta)')
plt.tight_layout()

### Where the named sensor sets sit

Overlay several named Bp sensor sets from `DATA/DIII-D/channel_filters.txt`, each set in its own
colour. The unrolled (phi, theta) panel shows the arrays as bands at distinct poloidal angles
(midplane, R±1, R±2, HFS); the R–z panel shows their radial placement against the wall.

In [ ]:
from omfit_compat import resolve_channel_filter

sets = ['Bp_LFS_midplane', 'Bp_LFS_R+1', 'Bp_LFS_R-1', 'Bp_LFS_R+2', 'Bp_LFS_R-2', 'Bp_HFS_Sensors']
colors = plt.cm.tab10.colors

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
handles = []
for i, name in enumerate(sets):
    regex = '|'.join(resolve_channel_filter(name, sd.device))  # friendly name -> single regex
    plots.plot_sensors(sd.raw, regex, geometry='cylindrical', ax=ax[0], color=colors[i])
    plots.plot_sensors(sd.raw, regex, geometry='rz',          ax=ax[1], color=colors[i])
    handles.append(plt.Line2D([], [], color=colors[i], label=name))

ax[0].set_title('Unrolled (phi–theta)')
ax[1].set_title('Cross-section (R–z)')
ax[0].legend(handles=handles, fontsize=8, loc='upper right')
plt.tight_layout()

## 3. Prep + fit

`run_steps` trims to the window with mode activity (2.5–4.2 s), causal band-passes 5–250 Hz,
SVD-conditions the data matrix (keeping 98% of the energy), and fits the n = 1, 2, 3 toroidal
harmonics. The reported **condition number K** is the central trust metric (warn K > 10, error
K > 20).

In [ ]:
r = run_steps(
    154551,
    channel_filter='Bp_LFS_midplane',
    ns=(1, 2, 3), ms=(0,),
    time_trim=(2.5, 4.2),
    prep_kwargs=dict(cutoff_hz=(5, 250), energy=0.98),
)
print(f'\ncondition number K = {r.condition_number:.2f}')
print(f'mean reduced chi^2 = {float(np.nanmean(r.fit["red_chi_sq"].values)):.1f}')
print(f'data-matrix SVD effective rank (98% energy) = {r.fit.attrs["signal_effective_rank"]} '
      f'of {r.fit.sizes["channel"]} sensors')

## 4. Signal conditioning

PREPARED traces (band-passed, SVD-cleaned) over the faint shifted RAW traces.

In [ ]:
plots.plot_signal(r.raw, r.prepared, 'MPID66M.*')
plt.gcf().set_size_inches(9, 4)

## 5. SVD conditioning (VISION's ≈ 98% energy)

**Top:** cumulative energy of the data-matrix singular values — a handful of coherent spatial
structures capture nearly all the energy (the rotating mode + its harmonics), the rest is noise.
**Bottom:** the design-matrix condition number per singular value, against the `fit_cond` cutoff.

In [ ]:
plots.plot_svds(r.fit)

## 6. Fit quality

Reduced chi^2, the signals, and the worst residuals. (chi^2 sits above 1 here because the
constant 2e-5 T sensor sigma is optimistic relative to unmodeled higher-n structure — the residuals
are still small compared with the signals.)

In [ ]:
plots.plot_fit(r.fit)

## 7. Mode amplitude & phase vs time — rotating → locking

The n = 1 amplitude (blue) grows through the window. Its phase winds rapidly early on (the mode is
**rotating**), then becomes coherent and slows (the mode **slows and locks**) — exactly the
tearing-mode dynamics VISION describes for this shot.

In [ ]:
plots.plot_fit_modes(r.fit)

## 8. The SLCONTOUR phi-vs-time contour

Reconstruct the fitted dB on a (toroidal angle) × time grid. A rotating mode shows **diagonal
stripes**; as it slows and locks the stripes straighten into vertical bands. White markers trace
the instantaneous peak (the mode's toroidal phase); the top panel is the RMS amplitude.

In [ ]:
plots.plot_slice(r.fit, fix_coord='theta', fix_value=0.0)

---
### Recap

We reproduced the VISION §4.1 SLCONTOUR outputs locally from the ported OMFIT scripts:
sensor map, conditioned signals, data/design-matrix SVD diagnostics, fit quality, **amplitude &
phase of each (n, m) mode vs time**, and the **phi-vs-time contour** showing the rotating n = 1
tearing mode that slows and locks. To analyse another array, change `channel_filter` (e.g.
`'Bp_LFS_R+1'`) and the `ns`/`ms` mode lists.